In [1]:
from pathlib import Path
import re
from pprint import pprint
from sentence_transformers import CrossEncoder
import numpy as np
import pandas as pd

In [2]:
# Questions 目录
QUESTIONS_DIR = Path("./Questions")

# ———————————————————— 参数：拆分用 ————————————————————
# 8 个固定 section
SECTION_TITLES = [
    "Problem Summary",
    "Formal Problem Definition",
    "Input Specification",
    "Output Specification",
    "Constraints",
    "Key Observations",
    "Algorithm Idea",
    "Detailed Algorithm Steps",
]

SECTION_VARS = {
    "Problem Summary": "ProblemSummary",
    "Formal Problem Definition": "FormalProblemDefinition",
    "Input Specification": "InputSpecification",
    "Output Specification": "OutputSpecification",
    "Constraints": "Constraints",
    "Key Observations": "KeyObservations",
    "Algorithm Idea": "AlgorithmIdea",
    "Detailed Algorithm Steps": "DetailedAlgorithmSteps",
}

FIRST_SECTION_MARK = "### [1] Problem Summary"

SECTION_PATTERN = re.compile(
    r"^###\s*\[(\d+)\]\s*(.+?)\s*$",
    re.MULTILINE
)

NUMBERED_STEP_RE = re.compile(r"^\s*\d+\.\s+")



# ———————————————————— 参数：CrossEncoder计算用 ————————————————————
model = CrossEncoder("cross-encoder/stsb-roberta-large")

# 块名顺序
KEY_ORDER = [
    "ProblemSummary",               # 1
    "FormalProblemDefinition",      # 2
    "InputSpecification",           # 3
    "OutputSpecification",          # 4
    "Constraints",                  # 5
    "KeyObservations",              # 6
    "AlgorithmIdea",                # 7
    "DetailedAlgorithmSteps",       # 8
    "Note",                         # 9
]

# 对应矩阵变量名
MAT_NAMES = [
    "mat1PS",
    "mat2FPD",
    "mat3IS",
    "mat4OS",
    "mat5C",
    "mat6KO",
    "mat7AI",
    "mat8DAS",
    "mat9N",
]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# 拆分用 函数部分
def list_question_dirs(questions_dir: Path):
    return sorted(
        [p for p in questions_dir.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda x: x.name
    )


def list_explain_txts(explain_dir: Path):
    txts = []

    for p in explain_dir.iterdir():
        if not (p.is_file() and p.suffix.lower() == ".txt"):
            continue

        parts = p.stem.split()

        if len(parts) >= 1 and parts[0].isdigit():
            txts.append((int(parts[0]), p))

    txts_sorted = sorted(txts, key=lambda x: x[0])
    display(txts_sorted)

    return [p for _, p in txts_sorted]


def remove_blank_lines(text: str) -> str:
    """
    对块内容做空行剔除：删除所有空行，仅保留非空行并按原顺序拼接。
    """
    if not text:
        return ""
    lines = [line for line in text.splitlines() if line.strip() != ""]
    return "\n".join(lines).strip()


def split_into_8_sections(text: str):
    """
    先切出 8 个块。
    舍弃 ### [1] Problem Summary 之前的任何内容。
    返回 dict:
    {
        'ProblemSummary': ...,
        ...
        'DetailedAlgorithmSteps': ...
    }
    """
    first_idx = text.find(FIRST_SECTION_MARK)
    if first_idx == -1:
        raise ValueError(f"未找到 '{FIRST_SECTION_MARK}'")

    text = text[first_idx:]
    matches = list(SECTION_PATTERN.finditer(text))
    if not matches:
        raise ValueError("未识别到 section 标题")

    parsed = {v: "" for v in SECTION_VARS.values()}

    for i, m in enumerate(matches):
        title = m.group(2).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end]

        if title in SECTION_VARS:
            parsed[SECTION_VARS[title]] = content

    return parsed


def extract_note_from_detailed(detailed_text: str):
    """
    从第八块中剥离第九块 Note。

    规则：
    1. 先对第八块原始内容处理，不提前删空行。
    2. 只考虑“最后一个由空行分隔出的连续非空块”是否应作为 Note。
    3. 若该最后块内部存在某行匹配 '数字. 空格'（如 5. xxx），
       则说明它属于第八块，不剥离。
    4. 若最后块内部没有编号行，且它前面由空行分隔，则将其剥离为 Note。
    5. 因为是按空行分块，若前面有编号步骤但中间已有空行断开，则最后块不再属于该步骤。
    """
    if not detailed_text.strip():
        return "", ""

    lines = detailed_text.splitlines()

    # 去掉末尾空行，便于找最后非空块
    end = len(lines) - 1
    while end >= 0 and lines[end].strip() == "":
        end -= 1

    if end < 0:
        return "", ""

    # 找最后一个连续非空块 [block_start, end]
    block_start = end
    while block_start >= 0 and lines[block_start].strip() != "":
        block_start -= 1
    block_start += 1

    # 如果整个 detailed 都是一个连续非空块，则不剥离 Note
    if block_start == 0:
        main_text = "\n".join(lines[:end + 1])
        return main_text, ""

    last_block_lines = lines[block_start:end + 1]

    # 若最后块内部含有编号步骤起始行，则它属于第八块
    if any(NUMBERED_STEP_RE.match(line) for line in last_block_lines):
        main_text = "\n".join(lines[:end + 1])
        return main_text, ""

    # 否则，最后块剥离为 Note
    note_text = "\n".join(last_block_lines)
    main_text = "\n".join(lines[:block_start - 1])  # block_start-1 是分隔空行

    return main_text, note_text


def parse_one_explain_file(txt_path: Path):
    """
    处理单个 explain 文件：
    1. 先切分出 8 个块
    2. 对第 8 块做第 9 块 Note 的剥离
    3. 对这 9 块内容统一做空行剔除
    返回：
    {
        'ProblemSummary': ...,
        ...
        'DetailedAlgorithmSteps': ...,
        'Note': ...
    }
    """
    raw = txt_path.read_text(encoding="utf-8")

    parsed8 = split_into_8_sections(raw)

    detailed_main, note_text = extract_note_from_detailed(parsed8["DetailedAlgorithmSteps"])
    parsed8["DetailedAlgorithmSteps"] = detailed_main

    parsed9 = dict(parsed8)
    parsed9["Note"] = note_text

    # 最后统一对 9 块做空行剔除
    for key in parsed9:
        parsed9[key] = remove_blank_lines(parsed9[key])

    return parsed9


def parse_question_dir(question_dir: Path):
    """
    对单个题目目录处理。
    返回：
    {
        'ProblemSummary': [file1内容, file2内容, ...],
        ...
        'DetailedAlgorithmSteps': [...],
        'Note': [...]
    }
    """
    explain_dir = question_dir / "LLM Explains"
    if not explain_dir.exists() or not explain_dir.is_dir():
        raise FileNotFoundError(f"{question_dir} 下不存在 'LLM Explains' 文件夹")

    txt_files = list_explain_txts(explain_dir)

    result = {v: [] for v in SECTION_VARS.values()}
    result["Note"] = []

    for txt_path in txt_files:
        parsed = parse_one_explain_file(txt_path)
        for key in result:
            result[key].append(parsed[key])

    return result

In [ ]:
# 遍历
dirs = list_question_dirs(QUESTIONS_DIR)
all_results = {}

for d in dirs:
    try:
        all_results[d.name] = parse_question_dir(d)
        print(f"done: {d.name}")
    except Exception as e:
        print(f"error: {d.name} -> {e}")

if all_results:
    first_name = next(iter(all_results))
    print(f"\n示例题目: {first_name}")
    pprint(all_results[first_name])

In [4]:
# 单独测试
dirs = list_question_dirs(QUESTIONS_DIR)
all_results = {}

d = dirs[0]
try:
    all_results[d.name] = parse_question_dir(d)
    print(f"done: {d.name}")
except Exception as e:
    print(f"error: {d.name} -> {e}")

if all_results:
    first_name = next(iter(all_results))
    print(f"\n示例题目: {first_name}")
    pprint(all_results[first_name])

[(1, PosixPath('Questions/Easy B3666/LLM Explains/1 0.0.txt')),
 (2, PosixPath('Questions/Easy B3666/LLM Explains/2 0.0.txt')),
 (3, PosixPath('Questions/Easy B3666/LLM Explains/3 0.0.txt')),
 (4, PosixPath('Questions/Easy B3666/LLM Explains/4 0.2.txt')),
 (5, PosixPath('Questions/Easy B3666/LLM Explains/5 0.2.txt')),
 (6, PosixPath('Questions/Easy B3666/LLM Explains/6 0.2.txt')),
 (7, PosixPath('Questions/Easy B3666/LLM Explains/7 0.5.txt')),
 (8, PosixPath('Questions/Easy B3666/LLM Explains/8 0.5.txt')),
 (9, PosixPath('Questions/Easy B3666/LLM Explains/9 0.5.txt')),
 (10, PosixPath('Questions/Easy B3666/LLM Explains/10 0.8.txt')),
 (11, PosixPath('Questions/Easy B3666/LLM Explains/11 0.8.txt')),
 (12, PosixPath('Questions/Easy B3666/LLM Explains/12 0.8.txt'))]

done: Easy B3666

示例题目: Easy B3666
{'AlgorithmIdea': ['- Use a stack or a similar data structure to keep track of '
                   'the indices of the suffix maximum values.\n'
                   '- When a new element is inserted, update the stack '
                   'accordingly to maintain the correct suffix maximum '
                   'values.\n'
                   '- Calculate the bitwise XOR of the indices of the suffix '
                   'maximum values after each insertion.',
                   '- Use a stack to keep track of the indices of the suffix '
                   'maximum values.\n'
                   '- When a new element is inserted, pop elements from the '
                   'stack that are smaller than the new element and update the '
                   'stack.\n'
                   '- Calculate the bitwise XOR of the indices in the stack.',
                   '- Use a stack to keep track of the indices of the suffix '
                   'maximum values.\n'


In [5]:
display(len(all_results["Easy B3666"]["AlgorithmIdea"]))
display(all_results["Easy B3666"]["AlgorithmIdea"][0])

12

'- Use a stack or a similar data structure to keep track of the indices of the suffix maximum values.\n- When a new element is inserted, update the stack accordingly to maintain the correct suffix maximum values.\n- Calculate the bitwise XOR of the indices of the suffix maximum values after each insertion.'

In [6]:
def build_similarity_matrix(text_list):
    n = len(text_list)
    mat = np.zeros((n, n))

    # 对角线 = 1
    for i in range(n):
        mat[i][i] = 1.0

    # 构造需要计算的 pairs（只算上三角）
    pairs = []
    index_pairs = []

    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((text_list[i], text_list[j]))
            index_pairs.append((i, j))

    if pairs:
        scores = model.predict(pairs)

        # 填充矩阵（对称）
        for (i, j), score in zip(index_pairs, scores):
            mat[i][j] = score
            mat[j][i] = score

    return mat


# ===== 主处理 =====

all_matrices = {}  # 每个题目对应9个矩阵

for qname, qdata in all_results.items():
    mats = {}

    for key, mat_name in zip(KEY_ORDER, MAT_NAMES):
        texts = qdata[key]
        mat = build_similarity_matrix(texts)
        mats[mat_name] = mat

    all_matrices[qname] = mats

In [8]:
TARGET_Q_INDEX = 0   # 指定打印 / 分析第几个题目（从 0 开始）
k = 3
temperature_list = [0.0, 0.2, 0.5, 0.8]

ROUND_DIGITS = 6

# 若只想看部分矩阵，可改成例如 ["mat8DAS", "mat9N"]
SELECTED_MATS = None

DEFAULT_MAT_NAMES = [
    "mat1PS",   # Problem Summary
    "mat2FPD",  # Formal Problem Definition
    "mat3IS",   # Input Specification
    "mat4OS",   # Output Specification
    "mat5C",    # Constraints
    "mat6KO",   # Key Observations
    "mat7AI",   # Algorithm Idea
    "mat8DAS",  # Detailed Algorithm Steps
    "mat9N",    # Note
]

def get_question_names_from_all_matrices(all_matrices):
    return list(all_matrices.keys())

def get_question_name_by_index(all_matrices, q_index):
    qnames = get_question_names_from_all_matrices(all_matrices)
    if not (0 <= q_index < len(qnames)):
        raise IndexError(f"q_index={q_index} 越界，当前题目总数为 {len(qnames)}")
    return qnames[q_index]

def print_question_matrices(all_matrices, q_index, selected_mats=None, round_digits=6):
    qname = get_question_name_by_index(all_matrices, q_index)
    mats = all_matrices[qname]

    mat_names = selected_mats if selected_mats is not None else list(mats.keys())

    print(f"题目 index = {q_index}")
    print(f"题目名称 = {qname}")

    for mat_name in mat_names:
        print(f"\n===== {mat_name} =====")
        mat = mats[mat_name]
        print(f"shape = {mat.shape}")
        display(pd.DataFrame(mat).round(round_digits))

# 直接打印指定题目的矩阵
print_question_matrices(
    all_matrices,
    q_index=TARGET_Q_INDEX,
    selected_mats=SELECTED_MATS,
    round_digits=ROUND_DIGITS
)

题目 index = 0
题目名称 = Easy B3666

===== mat1PS =====
shape = (12, 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,1.000000,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193,0.974160
1,0.974193,1.000000,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193,0.974160
2,0.974193,0.974193,1.000000,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193,0.974160
3,0.974193,0.974193,0.974193,1.000000,0.974193,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193,0.974160
4,0.974193,0.974193,0.974193,0.974193,1.000000,0.974193,0.974193,0.974193,0.974193,0.790381,0.974193,0.974160
5,0.974193,0.974193,0.974193,0.974193,0.974193,1.000000,0.974193,0.974193,0.974193,0.790381,0.974193,0.974160
6,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,1.000000,0.974193,0.974193,0.790381,0.974193,0.974160
7,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,1.000000,0.974193,0.790381,0.974193,0.974160
8,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,0.974193,1.000000,0.790381,0.974193,0.974160
9,0.790381,0.790381,0.790381,0.790381,0.790381,0.790381,0.790381,0.790381,0.790381,1.000000,0.800534,0.798766



===== mat2FPD =====
shape = (12, 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,1.000000,0.973901,0.973901,0.923796,0.973901,0.973901,0.860405,0.923796,0.923796,0.841818,0.973901,0.927145
1,0.973901,1.000000,0.973901,0.923796,0.973901,0.973901,0.860405,0.923796,0.923796,0.841818,0.973901,0.927145
2,0.973901,0.973901,1.000000,0.923796,0.973901,0.973901,0.860405,0.923796,0.923796,0.841818,0.973901,0.927145
3,0.923796,0.923796,0.923796,1.000000,0.930114,0.930114,0.829398,0.973013,0.973013,0.839651,0.930114,0.972425
4,0.973901,0.973901,0.973901,0.930114,1.000000,0.973901,0.860405,0.923796,0.923796,0.841818,0.973901,0.927145
5,0.973901,0.973901,0.973901,0.930114,0.973901,1.000000,0.860405,0.923796,0.923796,0.841818,0.973901,0.927145
6,0.860405,0.860405,0.860405,0.829398,0.860405,0.860405,1.000000,0.818847,0.818847,0.824820,0.858972,0.818902
7,0.923796,0.923796,0.923796,0.973013,0.923796,0.923796,0.818847,1.000000,0.973013,0.839651,0.930114,0.972425
8,0.923796,0.923796,0.923796,0.973013,0.923796,0.923796,0.818847,0.973013,1.000000,0.839651,0.930114,0.972425
9,0.841818,0.841818,0.841818,0.839651,0.841818,0.841818,0.824820,0.839651,0.839651,1.000000,0.849943,0.820447



===== mat3IS =====
shape = (12, 12)


...

,0,1,2,3,4,5,6,7,8,9,10,11
0,1.000000,0.818575,0.818575,0.838266,0.818575,0.818575,0.970616,0.932541,0.810673,0.866357,0.754130,0.739058
1,0.818575,1.000000,0.970888,0.845689,0.970888,0.970888,0.824118,0.833750,0.821872,0.813621,0.840039,0.686794
2,0.818575,0.970888,1.000000,0.845689,0.970888,0.970888,0.824118,0.833750,0.821872,0.813621,0.840039,0.686794
3,0.838266,0.845689,0.845689,1.000000,0.869233,0.869233,0.849064,0.848229,0.941775,0.856979,0.754508,0.699580
4,0.818575,0.970888,0.970888,0.869233,1.000000,0.970888,0.824118,0.833750,0.821872,0.813621,0.840039,0.686794
5,0.818575,0.970888,0.970888,0.869233,0.970888,1.000000,0.824118,0.833750,0.821872,0.813621,0.840039,0.686794
6,0.970616,0.824118,0.824118,0.849064,0.824118,0.824118,1.000000,0.932541,0.810673,0.866357,0.754130,0.739058
7,0.932541,0.833750,0.833750,0.848229,0.833750,0.833750,0.932541,1.000000,0.797845,0.877579,0.760276,0.747749
8,0.810673,0.821872,0.821872,0.941775,0.821872,0.821872,0.810673,0.797845,1.000000,0.846152,0.746930,0.689403
9,0.866357,0.813621,0.813621,0.856979,0.813621,0.813621,0.866357,0.877579,0.846152,1.000000,0.746599,0.716796



===== mat8DAS =====
shape = (12, 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,1.000000,0.769515,0.769515,0.720330,0.758832,0.758832,0.754559,0.714803,0.713924,0.744651,0.752694,0.726595
1,0.769515,1.000000,0.714528,0.682391,0.773932,0.773932,0.686904,0.699308,0.688461,0.683125,0.705305,0.729933
2,0.769515,0.714528,1.000000,0.682391,0.773932,0.773932,0.686904,0.699308,0.688461,0.683125,0.705305,0.729933
3,0.720330,0.682391,0.682391,1.000000,0.791889,0.791889,0.709921,0.831750,0.710677,0.742299,0.767787,0.791907
4,0.758832,0.773932,0.773932,0.791889,1.000000,0.941667,0.761130,0.815371,0.774957,0.755592,0.876943,0.816973
5,0.758832,0.773932,0.773932,0.791889,0.941667,1.000000,0.761130,0.815371,0.774957,0.755592,0.876943,0.816973
6,0.754559,0.686904,0.686904,0.709921,0.761130,0.761130,1.000000,0.675240,0.687916,0.667239,0.688756,0.683860
7,0.714803,0.699308,0.699308,0.831750,0.815371,0.815371,0.675240,1.000000,0.731211,0.746608,0.786537,0.775867
8,0.713924,0.688461,0.688461,0.710677,0.774957,0.774957,0.687916,0.731211,1.000000,0.708922,0.764034,0.752320
9,0.744651,0.683125,0.683125,0.742299,0.755592,0.755592,0.667239,0.746608,0.708922,1.000000,0.709590,0.733439



===== mat9N =====
shape = (12, 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,1.000000,0.411465,0.411465,0.696396,0.604229,0.604229,0.387355,0.673686,0.565645,0.387355,0.584945,0.612603
1,0.411465,1.000000,0.972024,0.611300,0.679835,0.679835,0.898351,0.593761,0.603485,0.898351,0.613833,0.566994
2,0.411465,0.972024,1.000000,0.611300,0.679835,0.679835,0.898351,0.593761,0.603485,0.898351,0.613833,0.566994
3,0.696396,0.611300,0.611300,1.000000,0.643723,0.643723,0.582730,0.915877,0.847119,0.582730,0.716842,0.731623
4,0.604229,0.679835,0.679835,0.643723,1.000000,0.943740,0.752545,0.669492,0.683269,0.752545,0.673984,0.664404
5,0.604229,0.679835,0.679835,0.643723,0.943740,1.000000,0.752545,0.669492,0.683269,0.752545,0.673984,0.664404
6,0.387355,0.898351,0.898351,0.582730,0.752545,0.752545,1.000000,0.508955,0.611040,0.970825,0.612079,0.521738
7,0.673686,0.593761,0.593761,0.915877,0.669492,0.669492,0.508955,1.000000,0.855786,0.558044,0.732685,0.715229
8,0.565645,0.603485,0.603485,0.847119,0.683269,0.683269,0.611040,0.855786,1.000000,0.599028,0.713458,0.694117
9,0.387355,0.898351,0.898351,0.582730,0.752545,0.752545,0.970825,0.558044,0.599028,1.000000,0.612079,0.521738


In [9]:
# =============================
# 三种分析方法：辅助函数
# =============================

import numpy as np
import pandas as pd

def check_matrix_and_build_groups(mat, k, temperature_list):
    n = mat.shape[0]
    expected_n = k * len(temperature_list)

    if mat.shape[0] != mat.shape[1]:
        raise ValueError(f"矩阵不是方阵，shape={mat.shape}")

    if n != expected_n:
        raise ValueError(
            f"矩阵大小与参数不匹配：matrix_n={n}, "
            f"但 k * len(temperature_list) = {k} * {len(temperature_list)} = {expected_n}"
        )

    # 顺序约定：
    # [0:k) -> temperature_list[0]
    # [k:2k) -> temperature_list[1]
    # ...
    temp_to_indices = {}
    for ti, temp in enumerate(temperature_list):
        start = ti * k
        end = start + k
        temp_to_indices[temp] = list(range(start, end))

    return temp_to_indices


def get_offdiag_values(mat):
    n = mat.shape[0]
    mask = ~np.eye(n, dtype=bool)
    return mat[mask]


def get_upper_triangle_values(submat):
    n = submat.shape[0]
    vals = []
    for i in range(n):
        for j in range(i + 1, n):
            vals.append(submat[i, j])
    return np.array(vals, dtype=float)


# -----------------------------
# 方法1：整体稳定性
# -----------------------------
def method1_global_stats(mat):
    vals = get_offdiag_values(mat)

    return {
        "offdiag_mean": float(np.mean(vals)),
        "offdiag_std": float(np.std(vals)),
        "offdiag_min": float(np.min(vals)),
        "offdiag_max": float(np.max(vals)),
    }


# -----------------------------
# 方法2：温度内 / 温度间分解
# -----------------------------
def method2_temperature_decomposition(mat, k, temperature_list):
    temp_to_indices = check_matrix_and_build_groups(mat, k, temperature_list)

    # 1) 每个温度内部（within-temp）
    within_rows = []
    within_all_vals = []

    for temp in temperature_list:
        idxs = temp_to_indices[temp]
        submat = mat[np.ix_(idxs, idxs)]
        vals = get_upper_triangle_values(submat)

        within_rows.append({
            "temperature": temp,
            "sample_indices": idxs,
            "within_mean": float(np.mean(vals)) if len(vals) > 0 else np.nan,
            "within_std": float(np.std(vals)) if len(vals) > 0 else np.nan,
            "within_min": float(np.min(vals)) if len(vals) > 0 else np.nan,
            "within_max": float(np.max(vals)) if len(vals) > 0 else np.nan,
        })

        within_all_vals.extend(vals.tolist())

    within_df = pd.DataFrame(within_rows)

    # 2) 不同温度之间（cross-temp）
    cross_mean = pd.DataFrame(index=temperature_list, columns=temperature_list, dtype=float)
    cross_std = pd.DataFrame(index=temperature_list, columns=temperature_list, dtype=float)

    cross_all_vals = []

    for i, temp_i in enumerate(temperature_list):
        idx_i = temp_to_indices[temp_i]

        for j, temp_j in enumerate(temperature_list):
            idx_j = temp_to_indices[temp_j]

            if i == j:
                cross_mean.loc[temp_i, temp_j] = np.nan
                cross_std.loc[temp_i, temp_j] = np.nan
                continue

            vals = []
            for a in idx_i:
                for b in idx_j:
                    vals.append(mat[a, b])

            vals = np.array(vals, dtype=float)
            cross_mean.loc[temp_i, temp_j] = float(np.mean(vals))
            cross_std.loc[temp_i, temp_j] = float(np.std(vals))

            # 只累计上三角温度对，避免重复
            if i < j:
                cross_all_vals.extend(vals.tolist())

    overall_within_mean = float(np.mean(within_all_vals)) if len(within_all_vals) > 0 else np.nan
    overall_cross_mean = float(np.mean(cross_all_vals)) if len(cross_all_vals) > 0 else np.nan

    summary = {
        "overall_within_mean": overall_within_mean,
        "overall_cross_mean": overall_cross_mean,
        "within_minus_cross": (
            overall_within_mean - overall_cross_mean
            if (not np.isnan(overall_within_mean) and not np.isnan(overall_cross_mean))
            else np.nan
        )
    }

    return {
        "within_df": within_df,
        "cross_mean_df": cross_mean,
        "cross_std_df": cross_std,
        "summary": summary,
    }


# -----------------------------
# 方法3：原型 / 离群分析
# -----------------------------
def method3_prototype_outlier(mat, k=None, temperature_list=None):
    n = mat.shape[0]

    row_mean_offdiag = (np.sum(mat, axis=1) - np.diag(mat)) / (n - 1)

    df = pd.DataFrame({
        "sample_index": np.arange(n),
        "row_mean_offdiag": row_mean_offdiag,
    })

    if k is not None and temperature_list is not None:
        expected_n = k * len(temperature_list)
        if expected_n == n:
            df["temperature"] = [temperature_list[i // k] for i in range(n)]
            df["local_index_in_temp"] = [i % k for i in range(n)]

    prototype_idx = int(np.argmax(row_mean_offdiag))
    outlier_idx = int(np.argmin(row_mean_offdiag))

    df = df.sort_values("row_mean_offdiag", ascending=False).reset_index(drop=True)

    return {
        "row_mean_df": df,
        "prototype_idx": prototype_idx,
        "prototype_score": float(row_mean_offdiag[prototype_idx]),
        "outlier_idx": outlier_idx,
        "outlier_score": float(row_mean_offdiag[outlier_idx]),
    }


# -----------------------------
# 汇总：单个矩阵三种分析
# -----------------------------
def analyze_one_matrix(mat, k, temperature_list):
    m1 = method1_global_stats(mat)
    m2 = method2_temperature_decomposition(mat, k, temperature_list)
    m3 = method3_prototype_outlier(mat, k=k, temperature_list=temperature_list)

    overview = {
        **m1,
        **m2["summary"],
        "prototype_idx": m3["prototype_idx"],
        "prototype_score": m3["prototype_score"],
        "outlier_idx": m3["outlier_idx"],
        "outlier_score": m3["outlier_score"],
    }

    return {
        "overview": overview,
        "method1": m1,
        "method2": m2,
        "method3": m3,
    }

In [10]:
# =============================
# 对指定题目的所有矩阵做分析
# =============================

def analyze_question_matrices(all_matrices, q_index, k, temperature_list, selected_mats=None, round_digits=6):
    qname = get_question_name_by_index(all_matrices, q_index)
    mats = all_matrices[qname]

    mat_names = selected_mats if selected_mats is not None else list(mats.keys())

    print(f"题目 index = {q_index}")
    print(f"题目名称 = {qname}")
    print(f"k = {k}")
    print(f"temperature_list = {temperature_list}")

    # 先做总表
    overview_rows = []

    all_results = {}

    for mat_name in mat_names:
        mat = mats[mat_name]
        result = analyze_one_matrix(mat, k, temperature_list)
        all_results[mat_name] = result

        overview_row = {"matrix": mat_name}
        overview_row.update(result["overview"])
        overview_rows.append(overview_row)

    overview_df = pd.DataFrame(overview_rows)
    print("\n===== 总览表 =====")
    display(overview_df.round(round_digits))

    # 再逐矩阵展开细节
    for mat_name in mat_names:
        result = all_results[mat_name]

        print(f"\n\n==============================")
        print(f"矩阵: {mat_name}")
        print(f"==============================")

        # 方法1
        print("\n[方法1] 整体稳定性（非对角元素统计）")
        display(pd.DataFrame([result["method1"]]).round(round_digits))

        # 方法2
        print("\n[方法2-A] 温度内稳定性")
        display(result["method2"]["within_df"].round(round_digits))

        print("\n[方法2-B] 温度间均值矩阵")
        display(result["method2"]["cross_mean_df"].round(round_digits))

        print("\n[方法2-C] 温度间标准差矩阵")
        display(result["method2"]["cross_std_df"].round(round_digits))

        print("\n[方法2-D] 温度分解汇总")
        display(pd.DataFrame([result["method2"]["summary"]]).round(round_digits))

        # 方法3
        print("\n[方法3] 原型 / 离群分析（按 row mean offdiag 排序）")
        display(result["method3"]["row_mean_df"].round(round_digits))

        print(
            f"prototype_idx = {result['method3']['prototype_idx']}, "
            f"prototype_score = {result['method3']['prototype_score']:.{round_digits}f}"
        )
        print(
            f"outlier_idx = {result['method3']['outlier_idx']}, "
            f"outlier_score = {result['method3']['outlier_score']:.{round_digits}f}"
        )

    return overview_df, all_results


overview_df, all_results = analyze_question_matrices(
    all_matrices=all_matrices,
    q_index=TARGET_Q_INDEX,
    k=k,
    temperature_list=temperature_list,
    selected_mats=SELECTED_MATS,
    round_digits=ROUND_DIGITS
)

题目 index = 0
题目名称 = Easy B3666
k = 3
temperature_list = [0.0, 0.2, 0.5, 0.8]

===== 总览表 =====


,matrix,offdiag_mean,offdiag_std,offdiag_min,offdiag_max,overall_within_mean,overall_cross_mean,within_minus_cross,prototype_idx,prototype_score,outlier_idx,outlier_score
0,mat1PS,0.943834,0.067888,0.790381,0.974193,0.945100,0.943552,0.001548,10,0.958403,9,0.792066
1,mat2FPD,0.914210,0.054109,0.818847,0.973901,0.913673,0.914329,-0.000657,10,0.935991,9,0.838478
2,mat3IS,0.877416,0.077569,0.805812,0.971522,0.879290,0.876999,0.002291,10,0.893812,9,0.836989
3,mat4OS,0.841974,0.101332,0.721734,0.974135,0.834751,0.843579,-0.008827,4,0.883699,7,0.761535
4,mat5C,0.972526,0.000000,0.972526,0.972526,0.972526,0.972526,0.000000,0,0.972526,0,0.972526
5,mat6KO,0.844349,0.103147,0.701450,0.973258,0.842972,0.844655,-0.001683,6,0.884403,10,0.724759
6,mat7AI,0.823220,0.079326,0.628432,0.970888,0.829190,0.821894,0.007296,4,0.856424,11,0.700659
7,mat8DAS,0.747226,0.053150,0.667239,0.941667,0.757891,0.744856,0.013035,4,0.803747,6,0.705778
8,mat9N,0.672008,0.134326,0.387355,0.972024,0.671502,0.672121,-0.000619,4,0.704327,0,0.539943




矩阵: mat1PS

[方法1] 整体稳定性（非对角元素统计）


,offdiag_mean,offdiag_std,offdiag_min,offdiag_max
0,0.943834,0.067888,0.790381,0.974193



[方法2-A] 温度内稳定性


...


[方法2-D] 温度分解汇总


,overall_within_mean,overall_cross_mean,within_minus_cross
0,0.671502,0.672121,-0.000619



[方法3] 原型 / 离群分析（按 row mean offdiag 排序）


,sample_index,row_mean_offdiag,temperature,local_index_in_temp
0,5,0.704327,0.2,2
1,4,0.704327,0.2,1
2,3,0.689397,0.2,0
3,9,0.684872,0.8,0
4,1,0.684476,0.0,1
5,2,0.684476,0.0,2
6,6,0.681501,0.5,0
7,7,0.680615,0.5,1
8,10,0.679091,0.8,1
9,8,0.678155,0.5,2


prototype_idx = 4, prototype_score = 0.704327
outlier_idx = 0, outlier_score = 0.539943
